In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.datasets import load_iris
import keras
from ucimlrepo import fetch_ucirepo 


ModuleNotFoundError: No module named 'ucimlrepo'

In [ ]:
# Capa personalizada que implementa una combinación de polinomios de Legendre
class PolynomialDense(tf.keras.layers.Layer):
    def __init__(self, units, degree=2, use_bias=True, **kwargs):
        super(PolynomialDense, self).__init__(**kwargs)
        self.units = units
        self.degree = degree
        self.use_bias = use_bias

    def build(self, input_shape):
        input_dim = input_shape[-1]

        self.kernels = []
        for d in range(1, self.degree + 1):
            w = self.add_weight(
                shape=(input_dim, self.units),
                initializer='glorot_uniform',
                trainable=True,
                name=f'kernel_degree_{d}'
            )
            self.kernels.append(w)

        if self.use_bias:
            self.bias = self.add_weight(
                shape=(self.units,),
                initializer='zeros',
                trainable=True,
                name='bias'
            )

    def call(self, inputs):

        #Asumimos que los inputs ya están normalizados en [-1, 1]
        x = inputs

        # --- 2. Generamos la base de Legendre
        # P0(x)
        P0 = tf.ones_like(x)

        if self.degree == 0:
            features = [P0]
        else:
            # P1(x)
            P1 = x
            features = [P1]

            if self.degree > 1:
                Pnm2 = P0
                Pnm1 = P1

                for n in range(2, self.degree + 1):
                    Pn = ((2.0 * n - 1.0) * x * Pnm1 - (n - 1.0) * Pnm2) / n
                    features.append(Pn)

                    Pnm2 = Pnm1
                    Pnm1 = Pn

        # --- 3. Combinación lineal con los kernels
        output = 0.0

        for phi, W in zip(features, self.kernels):
            output += tf.matmul(phi, W)

        if self.use_bias:
            output += self.bias

        return output


In [ ]:
#Ahora importaremos el dataSet de Iris


iris = load_iris()

X = iris.data
y = iris.target

In [ ]:
#Dividimos el dataset en entrenamiento y prueba
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

In [ ]:
X.shape, y.shape

((150, 4), (150,))

In [ ]:
#Normalizamos los datos
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
input_dim = X_train.shape[1]

#Modelo lineal
inputLinear = keras.Input(shape=(input_dim,))
x1 = layers.Dense(32, activation='relu')(inputLinear)
x2 = layers.Dense(16, activation='relu')(x1)
outputLinear = layers.Dense(3, activation='softmax')(x2)

#MODELO DE RED POLINÓMICA DE LEGENDRE

#Modelo polinómico grado 2

inputPoli = keras.Input(shape=(input_dim,))

x = PolynomialDense(32, degree=2)(inputPoli)
x = layers.Activation('tanh')(x)
x = layers.Dense(16, activation='tanh')(x)

outputPoli = layers.Dense(3, activation='softmax')(x)

#Modelo polinómico grado 3
inputPoli3 = keras.Input(shape=(input_dim,))
x3 = PolynomialDense(32, degree=3)(inputPoli3)
x3 = layers.Activation('tanh')(x3)
x3 = layers.Dense(16, activation='tanh')(x3)
outputPoli3 = layers.Dense(3, activation='softmax')(x3)

#Modelo polinómico grado 4
inputPoli4 = keras.Input(shape=(input_dim,))
x4 = PolynomialDense(32, degree=4)(inputPoli4)
x4 = layers.Activation('tanh')(x4)
x4 = layers.Dense(16, activation='tanh')(x4)
outputPoli4 = layers.Dense(3, activation='softmax')(x4)

In [ ]:
#Guardamos ambos modelos
modelLinear  = keras.Model(inputs=inputLinear, outputs=outputLinear)
modelPoli = keras.Model(inputs=inputPoli, outputs=outputPoli) 
modelPoli3 = keras.Model(inputs=inputPoli3, outputs=outputPoli3)
modelPoli4 = keras.Model(inputs=inputPoli4, outputs=outputPoli4)



In [ ]:
modelPoli3.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ polynomial_dense_1              │ (None, 32)             │           416 │
│ (PolynomialDense)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 995 (3.89 KB)

 Trainable params: 995 (3.89 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Compilamos ambos modelos
modelLinear.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelPoli4.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
#Añadimos Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
#Entrenamos el modelo
historyLinear = modelLinear.fit(X_train, y_train, epochs=100, batch_size=16, callbacks=[early_stopping], validation_data=(X_test, y_test))
historyPoli = modelPoli.fit(X_train, y_train, epochs=100, batch_size=16, callbacks=[early_stopping], validation_data=(X_test, y_test))
historyPoli3 = modelPoli3.fit(X_train, y_train, epochs=100, batch_size=16, callbacks=[early_stopping], validation_data=(X_test, y_test))
historyPoli4 = modelPoli4.fit(X_train, y_train, epochs=100, batch_size=16, callbacks=[early_stopping], validation_data=(X_test, y_test))

Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.3591 - loss: 1.2358 - val_accuracy: 0.3111 - val_loss: 1.2273
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3288 - loss: 1.1944 - val_accuracy: 0.2889 - val_loss: 1.1879
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3019 - loss: 1.1624 - val_accuracy: 0.2889 - val_loss: 1.1552
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2158 - loss: 1.1274 - val_accuracy: 0.4222 - val_loss: 1.1268
Epoch 5/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4646 - loss: 1.0980 - val_accuracy: 0.5556 - val_loss: 1.0990
Epoch 6/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6781 - loss: 1.0586 - val_accuracy: 0.6000 - val_loss: 1.0735
Epoch 7/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7183 - loss: 1.0279 - val_accuracy: 0.6000 - val_loss: 1.0487
Epoch 8/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6730 - loss: 1.0124 - val_accuracy: 0.6000 - val_loss: